In [64]:
# Preprocessing

df = pd.read_csv(path + '/train.csv')
X = df.copy()
# Use the US population mean to replace missing sleep duration values
X['sleep_duration'] = X['sleep_duration'].mask(X['sleep_duration'].isnull(), other=7.0)
# Because of high variability of heart rate statistics, use the 
# sample mean for heart_rate
X['heart_rate'] = X['heart_rate'].mask(X['heart_rate'].isnull(), other=X['sleep_duration'].mean())
# https://dqydj.com gives a median bmi of 25.5 for 18-24 year olds
X['bmi'] = X['bmi'].mask(X['bmi'].isnull(), other=25.5)
# The following seem like local statistics, so I'll take the mean of the sample'
# Using median for quantitative variables and mode for categorical
X['calorie_expenditure'] = X['calorie_expenditure'].mask(X['calorie_expenditure'].isnull(), other=X['calorie_expenditure'].median())
X['step_count'] = X['step_count'].mask(X['step_count'].isnull(), other=X['step_count'].median())
X['exercise_duration'] = X['exercise_duration'].mask(X['exercise_duration'].isnull(), other=X['exercise_duration'].median())
X['water_intake'] = X['water_intake'].mask(X['water_intake'].isnull(), other=X['water_intake'].median())
X['diet_type'] = X['diet_type'].mask(X['diet_type'].isnull(), other=X['diet_type'].mode()[0])
X['stress_level'] = X['stress_level'].mask(X['stress_level'].isnull(), other=X['stress_level'].mode()[0])
X['sleep_quality'] = X['sleep_quality'].mask(X['sleep_quality'].isnull(), other=X['sleep_quality'].mode()[0])
X['physical_activity_level'] = X['physical_activity_level'].mask(X['physical_activity_level'].isnull(), X['physical_activity_level'].mode()[0])
# I'll take a pessimistic view and assume that unreported smoking/drinking means 'yes'
X['smoking_alcohol'] = X['smoking_alcohol'].mask(X['smoking_alcohol'].isnull(), other='yes')
# Create a third category, "other", for null gender values
X['gender'] = X['gender'].mask(X['gender'].isnull(), other="other")

X_with_categories = X.copy()
X = X.drop(columns=['id'])
categorical = ['diet_type', 'smoking_alcohol', 'gender']

one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

for category in categorical:
    # encode
    new_column_data = one_hot.fit_transform(X[[category]])
    # get names for the one-hot-encoded columns
    new_column_names = one_hot.get_feature_names_out([category])
    # wrap in dataframe
    new_columns = pd.DataFrame(new_column_data, columns=new_column_names)
    # attach to the main dataframe
    X = pd.concat([X.drop(columns=category, axis=1), new_columns], axis=1)
    
ordinal = ['health_condition', 'sleep_quality', 'stress_level', 'physical_activity_level']

ordinal_values = {}
ordinal_values['health_condition'] = ['unhealthy', 'at-risk', 'fit']
ordinal_values['sleep_quality'] = ['poor', 'average', 'good']
ordinal_values['stress_level'] = ['low', 'medium', 'high']
ordinal_values['physical_activity_level'] = ['sedentary', 'moderate', 'active']

for category in ordinal:
    encoder = OrdinalEncoder(categories=[ordinal_values[category]])
    X[category + '_encoded'] = encoder.fit_transform(X[[category]])
    X.drop(columns=[category], inplace=True)

## Feature engineering

Below I create some new features based on initial hypotheses and patterns observed in EDA. First, though, I'll split the training and test set and normalize the data to make it easier to compare and combine features.

In [65]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

In [66]:
train_X.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type_balanced,diet_type_non-veg,diet_type_veg,smoking_alcohol_no,smoking_alcohol_occasional,smoking_alcohol_yes,gender_female,gender_male,gender_other,health_condition_encoded,sleep_quality_encoded,stress_level_encoded,physical_activity_level_encoded
652972,7.76,80.9,22.51,1828.0,4368.0,35.3,2.38,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,2.0,0.0,1.0
226392,4.58,86.8,23.45,2241.0,6183.0,52.0,1.19,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,2.0
651569,9.32,75.6,23.19,2539.0,7648.0,25.8,2.05,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,2.0,0.0,0.0
47990,6.62,68.9,23.68,2075.0,12409.0,56.6,2.84,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0
582945,6.61,85.7,23.06,2123.0,11859.0,62.9,2.70,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,2.0,2.0,2.0


In [67]:
nonnormal = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 
             'step_count', 'exercise_duration', 'water_intake']

for cat in nonnormal:
    train_X[cat + '_norm'] = train_X[cat] / train_X[cat].max()
    val_X[cat + '_norm'] = val_X[cat] / val_X[cat].max()

In [68]:
# Subtract physical activity (maybe independent) from sleep_duration, sleep_quality, 
# and stress_level. First multiply physical activity by some contribution factor

contribution_factor = 0.3
train_X['sleep_duration_no_activity'] = train_X['sleep_duration_norm'] - contribution_factor * train_X['physical_activity_level_encoded']
train_X['sleep_quality_encoded_no_activity'] = train_X['sleep_quality_encoded'] - contribution_factor * train_X['physical_activity_level_encoded']
train_X['stress_level_no_activity'] = train_X['stress_level_encoded'] - contribution_factor * train_X['physical_activity_level_encoded']


val_X['sleep_duration_no_activity'] = val_X['sleep_duration_norm'] - contribution_factor * val_X['physical_activity_level_encoded']
val_X['sleep_quality_encoded_no_activity'] = val_X['sleep_quality_encoded'] - contribution_factor * val_X['physical_activity_level_encoded']
val_X['stress_level_no_activity'] = val_X['stress_level_encoded'] - contribution_factor * val_X['physical_activity_level_encoded']

In [69]:
train_X['sleep_duration_no_activity'].describe()

count    517566.000000
mean          0.402276
std           0.263260
min          -0.300000
25%           0.181000
50%           0.400000
75%           0.630000
max           1.000000
Name: sleep_duration_no_activity, dtype: float64

In [70]:
train_X['sleep_quality_encoded_no_activity'].describe()

count    517566.000000
mean          0.693247
std           0.813339
min          -0.600000
25%           0.000000
50%           0.700000
75%           1.400000
max           2.000000
Name: sleep_quality_encoded_no_activity, dtype: float64

In [71]:
train_X['stress_level_no_activity'].describe()

count    517566.000000
mean          0.717358
std           0.749982
min          -0.600000
25%           0.400000
50%           0.700000
75%           1.400000
max           2.000000
Name: stress_level_no_activity, dtype: float64

In [72]:
# The 'fit' category seemed to have higher step counts and exercise 
# duration than the other groups

train_X['step_count_high'] = (train_X['step_count'] > 9000)
train_X['exercise_duration_high'] = (train_X['exercise_duration'] > 45)

val_X['step_count_high'] = (val_X['step_count'] > 9000)
val_X['exercise_duration_high'] = (val_X['exercise_duration'] > 45)

In [73]:
# A number of features seem to change when the heart rate is over 68 
# and again when it is over 78.

train_X['bmi_high_hr'] = train_X['bmi'].where(train_X['heart_rate']<=68, other=2*train_X['bmi'])
train_X['smoking_high_hr'] = train_X['smoking_alcohol_yes'].where(train_X['heart_rate']<=68, other=2*train_X['smoking_alcohol_yes'])
train_X['sleep_qual_high_hr'] = train_X['sleep_quality_encoded'].where(train_X['heart_rate']>=68, other=2*train_X['sleep_quality_encoded'])


val_X['bmi_high_hr'] = val_X['bmi'].where(val_X['heart_rate']<=68, other=2*val_X['bmi'])
val_X['smoking_high_hr'] = val_X['smoking_alcohol_yes'].where(val_X['heart_rate']<=68, other=2*val_X['smoking_alcohol_yes'])
val_X['sleep_qual_high_hr'] = val_X['sleep_quality_encoded'].where(val_X['heart_rate']>=68, other=2*val_X['sleep_quality_encoded'])

In [74]:
# sleep quality, stress level, and smoking/alcohol for low and moderate 
# activity levels, replaced by high activity level where present
train_X['sleep_quality_active'] = train_X['sleep_quality_encoded'].where(train_X['physical_activity_level_encoded'] <= 1, other=train_X['physical_activity_level_encoded'])
train_X['stress_level_active'] = train_X['stress_level_encoded'].where(train_X['physical_activity_level_encoded'] <= 1, other=train_X['physical_activity_level_encoded'])
train_X['smoking_active'] = train_X['smoking_alcohol_no'].where(train_X['physical_activity_level_encoded'] <= 1, other=train_X['physical_activity_level_encoded'])


val_X['sleep_quality_active'] = val_X['sleep_quality_encoded'].where(val_X['physical_activity_level_encoded'] <= 1, other=val_X['physical_activity_level_encoded'])
val_X['stress_level_active'] = val_X['stress_level_encoded'].where(val_X['physical_activity_level_encoded'] <= 1, other=val_X['physical_activity_level_encoded'])
val_X['smoking_active'] = val_X['smoking_alcohol_no'].where(val_X['physical_activity_level_encoded'] <= 1, other=val_X['physical_activity_level_encoded'])

train_X.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type_balanced,diet_type_non-veg,diet_type_veg,...,sleep_quality_encoded_no_activity,stress_level_no_activity,step_count_high,exercise_duration_high,bmi_high_hr,smoking_high_hr,sleep_qual_high_hr,sleep_quality_active,stress_level_active,smoking_active
652972,7.76,80.9,22.51,1828.0,4368.0,35.3,2.38,0.0,0.0,1.0,...,1.7,-0.3,False,False,45.02,2.0,2.0,2.0,0.0,0.0
226392,4.58,86.8,23.45,2241.0,6183.0,52.0,1.19,1.0,0.0,0.0,...,-0.6,-0.6,False,True,46.90,2.0,0.0,2.0,2.0,2.0
651569,9.32,75.6,23.19,2539.0,7648.0,25.8,2.05,1.0,0.0,0.0,...,2.0,0.0,False,False,46.38,0.0,2.0,2.0,0.0,0.0
47990,6.62,68.9,23.68,2075.0,12409.0,56.6,2.84,1.0,0.0,0.0,...,-0.6,-0.6,True,True,47.36,0.0,0.0,2.0,2.0,2.0
582945,6.61,85.7,23.06,2123.0,11859.0,62.9,2.70,0.0,0.0,1.0,...,1.4,1.4,True,True,46.12,2.0,2.0,2.0,2.0,2.0


In [75]:
#Train a new model

rf_model = RandomForestClassifier(n_estimators=100, max_depth=9)
rf_model.fit(train_X, train_y)
val_predictions = rf_model.predict(val_X)
print(f"Balanced accuracy score with engineered features: {balanced_accuracy_score(val_y, val_predictions)}")

Balanced accuracy score with engineered features: 0.996803284204859


In [77]:
from sklearn.model_selection import StratifiedKFold
# The score above seems way too high. Let's try cross-fold validation
# on the training set

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
scores = cross_val_score(rf_model, train_X, train_y, cv=skf, scoring='balanced_accuracy')

In [78]:
print(f"cross val scores: {scores}")

cross val scores: [0.99864345 0.99228033 0.99798926 0.99654404 0.9918791 ]
